<a href="https://colab.research.google.com/github/asdq11870-cyber/PyTorch/blob/main/Testing%20Pretrained%20Model%20EfficientNetB2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Model Deployment

In [ ]:
import matplotlib.pyplot as plt
import torch
import torchvision
from torch import nn
from torchvision import transforms
device = "cuda" if torch.cuda.is_available() else "cpu"

try:
    from torchinfo import summary
except:
    print("Could not find torchinfo. Installing...")
    !pip install -q torchinfo
    from torchinfo import summary

try:
    from going_modular import data_setup, engine, model_builder, transformations, utils, predictions, dataset_builder
except:
    !git clone https://github.com/asdq11870-cyber/PyTorch
    !mv PyTorch/going_modular .
    !mv PyTorch/assets .
    !mv PyTorch/models .
    !rm -rf PyTorch
    from going_modular import data_setup, engine, model_builder, transformations, utils, predictions, dataset_builder

Could not find torchinfo. Installing...
Cloning into 'PyTorch'...
remote: Enumerating objects: 298, done.
remote: Counting objects: 100% (88/88), done.
remote: Compressing objects: 100% (63/63), done.
remote: Total 298 (delta 35), reused 41 (delta 14), pack-reused 210 (from 1)
Receiving objects: 100% (298/298), 251.63 KiB | 3.14 MiB/s, done.
Resolving deltas: 100% (160/160), done.
Filtering content: 100% (4/4), 344.46 MiB | 13.10 MiB/s, done.


In [ ]:
extract_path = utils.download_data(
    source="https://github.com/asdq11870-cyber/PyTorch/raw/refs/heads/main/data/watch_shoe_fragrance.zip",
    destination="watch_shoe_fragrance",
    remove_source=True
)

Unzipping


In [ ]:
weights = torchvision.models.EfficientNet_B2_Weights.DEFAULT
auto_transform = weights.transforms()

test_transform = transformations.create_data_transforms(
    height=224, width=224, normalise=True,
    mean=[0.5,0.5,0.5], std=[0.5,0.5,0.5]
)
train_dataloader, val_dataloader, test_dataloader, class_names = data_setup.create_dataloaders(
    train_dir=extract_path/"train",
    test_dir=extract_path/"test",
    val_dir=extract_path/"val",
    train_transform=auto_transform,
    test_and_val_transform=test_transform,
    batch_size=32
)
print(class_names)

['fragrance', 'shoe', 'watch']


In [ ]:
model0 = torchvision.models.efficientnet_b2(weights=weights)

for param in model0.features.parameters():
  param.requires_grad = False

for param in model0.features[-2].parameters():
  param.requires_grad = True

model0.classifier = nn.Sequential(
    nn.Dropout(p=0.3,inplace=True),
    nn.Linear(in_features=model0.classifier[1].in_features,out_features=len(class_names))
)

model0.to(device)
summary(
    model=model0,
    input_size=(1,3,224,224),
    col_names=["input_size","output_size","num_params","trainable"],
    col_width=20,
    row_settings=["var_names"]
)


Downloading: "https://download.pytorch.org/models/efficientnet_b2_rwightman-c35c1473.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_b2_rwightman-c35c1473.pth


100%|██████████| 35.2M/35.2M [00:00<00:00, 42.6MB/s]


Layer (type (var_name))                                      Input Shape          Output Shape         Param #              Trainable
EfficientNet (EfficientNet)                                  [1, 3, 224, 224]     [1, 3]               --                   Partial
├─Sequential (features)                                      [1, 3, 224, 224]     [1, 1408, 7, 7]      --                   Partial
│    └─Conv2dNormActivation (0)                              [1, 3, 224, 224]     [1, 32, 112, 112]    --                   False
│    │    └─Conv2d (0)                                       [1, 3, 224, 224]     [1, 32, 112, 112]    (864)                False
│    │    └─BatchNorm2d (1)                                  [1, 32, 112, 112]    [1, 32, 112, 112]    (64)                 False
│    │    └─SiLU (2)                                         [1, 32, 112, 112]    [1, 32, 112, 112]    --                   --
│    └─Sequential (1)                                        [1, 32, 112, 112]    [1,

In [ ]:
writer0 = utils.create_writer(
    experiment_name="x",
    model_name="effnetb2",
    extra="10_epochs"
)

[INFO] Created SummaryWriter, saving to: runs/13-07-2026/x/effnetb2/10_epochs...


In [ ]:
optimiser0 = torch.optim.Adam(params=model0.parameters(), lr=1e-3)
engine.batch_train(
    model0, train_dataloader, val_dataloader, test_dataloader, 10, 1, device, False,
    optimiser0, torch.nn.CrossEntropyLoss(), writer0, torch.optim.lr_scheduler.CosineAnnealingLR(optimiser0, 10, 1e-4), "effnetb2.pth", "models"
)

Epoch: 1 
 ---------------------------------------------------------

Train Loss: 0.14405 | Train Accuracy: 96.17% | Validation Loss: 0.02794 | Validation Accuracy: 100.00%

Epoch: 2 
 ---------------------------------------------------------

Train Loss: 0.00979 | Train Accuracy: 99.92% | Validation Loss: 0.02390 | Validation Accuracy: 100.00%

Epoch: 3 
 ---------------------------------------------------------

Train Loss: 0.00467 | Train Accuracy: 99.83% | Validation Loss: 0.01257 | Validation Accuracy: 100.00%

Epoch: 4 
 ---------------------------------------------------------

Train Loss: 0.00181 | Train Accuracy: 100.00% | Validation Loss: 0.00691 | Validation Accuracy: 100.00%

Epoch: 5 
 ---------------------------------------------------------


KeyboardInterrupt: 

In [ ]:
list_of_preds = predictions.pred_and_store(extract_path/"test", model0, auto_transform, class_names, device)
from pprint import pprint
pprint(list_of_preds[:5])

In [ ]:
# weights = torchvision.models.ViT_B_16_Weights.DEFAULT
# auto_transform = weights.transforms()

# train_dataloader, val_dataloader, test_dataloader, class_names = data_setup.create_dataloaders(
#     train_dir=extract_path/"train",
#     test_dir=extract_path/"test",
#     val_dir=extract_path/"val",
#     train_transform=auto_transform,
#     test_and_val_transform=test_transform,
#     batch_size=32
# )

In [ ]:
# model1 = torchvision.models.vit_b_16(weights=weights)

# for params in model1.conv_proj.parameters():
#   params.requires_grad = False

# for params in model1.heads.parameters():
#   params.requires_grad = True

# for params in model1.encoder.layers.parameters():
#   params.requires_grad = True

# model1.heads = nn.Sequential(
#     nn.Linear(in_features=768, out_features=len(class_names))
# )

# model1.to(device)
# summary(
#     model=model1,
#     input_size=(1,3,224,224),
#     col_names=["input_size","output_size","num_params","trainable"],
#     col_width=20,
#     row_settings=["var_names"]
# )

In [ ]:
# writer1 = utils.create_writer(
#     experiment_name="x", model_name="ViT", extra="10_epochs"
# )

In [ ]:
# optimiser1 = torch.optim.AdamW(
#     params=filter(lambda p: p.requires_grad, model1.parameters()),
#     lr=1e-5,
#     weight_decay=1e-4
# )
# engine.batch_train(
#     model1, train_dataloader, val_dataloader, test_dataloader, 10, 1, device, False,
#     optimiser1, torch.nn.CrossEntropyLoss(), writer1, torch.optim.lr_scheduler.CosineAnnealingLR(optimiser1, 10, 1e-6), "ViT.pth", "models"
# )

In [ ]:
import gradio as gr
gr.__version__

In [ ]:
model0.to("cpu")
next(iter(model0.parameters())).device

In [ ]:
from typing import Tuple, Dict
from timeit import default_timer as timer

def predict(img) -> Tuple[Dict, float]:
    """Transforms and performs a prediction on img and returns prediction and time taken.
    """
    # Start the timer
    start_time = timer()

    # Transform the target image and add a batch dimension
    img = auto_transform(img).unsqueeze(0)

    # Put model into evaluation mode and turn on inference mode
    model0.eval()
    with torch.inference_mode():
        # Pass the transformed image through the model and turn the prediction logits into prediction probabilities
        pred_probs = torch.softmax(model0(img), dim=1)

    # Create a prediction label and prediction probability dictionary for each prediction class (this is the required format for Gradio's output parameter)
    pred_labels_and_probs = {class_names[i]: float(pred_probs[0][i]) for i in range(len(class_names))}

    # Calculate the prediction time
    pred_time = round(timer() - start_time, 5)

    # Return the prediction dictionary and prediction time
    return pred_labels_and_probs, pred_time

In [ ]:
try:
  model0.load_state_dict(torch.load(f="models/effnetb2.pth", map_location=device))
except Exception:
  print("Could not find file!")

In [ ]:
title = "Watch Shoe Fragrance Determinator!"
description = "Using an EfficientNetB2 feature extraction model"
demo = gr.Interface(
    fn=predict,
    inputs=gr.Image(type="pil"),
    outputs=[gr.Label(num_top_classes=3, label="Predictions"), gr.Number(label="Prediction Time (s)")],
    title=title, description=description
)
demo.launch(debug=False, share=True)

In [ ]:
# import shutil
# from pathlib import Path
# import random
# wsf_path = Path("demo/watch_shoe_fragrance")
# if wsf_path.exists():
#   shutil.rmtree(wsf_path)
# wsf_path.mkdir(parents=True, exist_ok=True)
# wsf_examples_path = wsf_path / "examples"
# wsf_examples_path.mkdir(parents=True, exist_ok=True)
# examples = [Path(filepath) for filepath in random.sample(list(Path(extract_path/"test").glob("*/*.jpg")),k=3)]
# for example in examples:
#   destination = wsf_examples_path / example.name
#   print(f"Copying {example} to {destination}")
#   shutil.copy2(src=example, dst=destination)
# shutil.copy2(src="models/effnetb2.pth",dst=wsf_path)

In [ ]:
# %%writefile demo/watch_shoe_fragrance/model.py
# import torch
# import torchvision
# from torch import nn
# def create_effnetb2_model(seed:int=32,num_classes:int=3):
#   weights = torchvision.models.EfficientNet_B2_Weights.DEFAULT
#   transform = weights.transforms()
#   model = torchvision.models.efficientnet_b2(weights=weights)

#   for param in model.features.parameters():
#     param.requires_grad = False

#   for param in model.features[-2].parameters():
#     param.requires_grad = True

#   torch.manual_seed(seed)
#   model.classifier = nn.Sequential(
#       nn.Dropout(p=0.3,inplace=True),
#       nn.Linear(in_features=model.classifier[1].in_features,out_features=num_classes)
#   )

#   return model, transform


In [ ]:
# %%writefile demo/watch_shoe_fragrance/app.py
# import gradio as gr
# import torch
# import os
# from model import create_effnetb2_model
# from typing import Tuple, Dict
# from timeit import default_timer as timer

# class_names = ["fragrance", "shoe", "watch"]

# effnetb2, effnetb2_transform = create_effnetb2_model(num_classes=len(class_names))
# effnetb2.load_state_dict(torch.load(f="effnetb2.pth",map_location=torch.device("cpu")))

# def predict(img) -> Tuple[Dict, float]:
#     """
#     Transforms and performs a prediction on img and returns prediction and time taken.
#     """
#     start_time = timer()

#     img = effnetb2_transform(img).unsqueeze(0)

#     effnetb2.eval()
#     with torch.inference_mode():
#         pred_probs = torch.softmax(effnetb2(img), dim=1)

#     pred_labels_and_probs = {class_names[i]: float(pred_probs[0][i]) for i in range(len(class_names))}

#     pred_time = round(timer() - start_time, 5)

#     return pred_labels_and_probs, pred_time

# example_list = [["examples/" + example] for example in os.listdir("examples")]
# title = "Watch Shoe Fragrance Determinator!"
# description = "Using an EfficientNetB2 feature extraction model"
# demo = gr.Interface(
#     fn=predict,
#     inputs=gr.Image(type="pil"),
#     outputs=[gr.Label(num_top_classes=3, label="Predictions"), gr.Number(label="Prediction Time (s)")],
#     title=title, description=description,examples=example_list
# )
# demo.launch(debug=False, share=True)


In [ ]:
print(torch.__version__)
print(torchvision.__version__)
print(gr.__version__)

In [ ]:
# %%writefile demo/watch_shoe_fragrance/requirements.txt
# torch==2.11.0
# torchvision==0.26.0
# gradio==6.19.0

In [ ]:
# !ls demo/watch_shoe_fragrance/

In [ ]:
# !cd demo/watch_shoe_fragrance && zip -r ../wsf_huggingface.zip * -x "*.pyc" "*.ipynb" "*__pycache__*" "*ipynb_checkpoints*"